# Documentation Evaluation & Benchmarking

This notebook explores the HIVE-AGENT evaluation stack:
- BERTScore + ROUGE-based documentation quality scoring
- Docstring coverage analysis
- Quality rubric grading
- Benchmark runner for comparing generation strategies

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 1. Docstring Coverage Analyzer

In [ ]:
from evaluation.coverage_analyzer import CoverageAnalyzer

code_with_docs = '''
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b  # no docstring

class Calculator:
    """A simple calculator class."""

    def divide(self, a: float, b: float) -> float:
        # no docstring
        return a / b
'''

analyzer = CoverageAnalyzer()
coverage = analyzer.analyze(code_with_docs)
print('Coverage report:', coverage)

df_cov = pd.DataFrame([
    {'Symbol': k, 'Has Docstring': v}
    for k, v in coverage.get('symbols', {}).items()
])
print(df_cov)

## 2. Quality Rubric

In [ ]:
from evaluation.quality_rubric import QualityRubric

rubric = QualityRubric()

docstrings = [
    """Add two numbers.

    Args:
        a (int): First operand.
        b (int): Second operand.

    Returns:
        int: Sum of a and b.

    Example:
        >>> add(1, 2)
        3
    """,
    "Add a and b.",
    "",
]

labels = ['Excellent', 'Minimal', 'Empty']
scores = [rubric.grade(d) for d in docstrings]
for label, score in zip(labels, scores):
    print(f'{label}: {score}')

In [ ]:
# Radar chart of quality dimensions
if scores and isinstance(scores[0], dict):
    dims = list(scores[0].keys())
    fig = go.Figure()
    for label, score in zip(labels, scores):
        vals = [score.get(d, 0) for d in dims]
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]], theta=dims + [dims[0]],
            fill='toself', name=label
        ))
    fig.update_layout(title='Documentation Quality Radar', polar=dict(radialaxis=dict(visible=True, range=[0, 1])))
    fig.show()
else:
    print('Quality scores:', scores)

## 3. Doc Evaluator (BERTScore + ROUGE)

In [ ]:
from evaluation.doc_evaluator import DocEvaluator

evaluator = DocEvaluator()

references = [
    "Return the sum of two integers a and b.",
    "Authenticate a user given username and password.",
]
hypotheses = [
    "Compute the addition of a and b and return the result.",
    "Check if the user credentials are valid.",
]

eval_results = evaluator.evaluate(hypotheses, references)
print('Evaluation results:', eval_results)

## 4. Benchmark Runner

In [ ]:
from evaluation.benchmark_runner import BenchmarkRunner

benchmark = BenchmarkRunner()

# Simulated generation strategies
strategies = {
    'template':  ['Add two numbers.', 'Authenticate user.'],
    'llm_short': ['Return sum of a and b.', 'Verify user credentials.'],
    'llm_full':  [
        'Return the sum of two integers a and b.',
        'Authenticate a user given username and password.',
    ],
}
refs = [
    'Return the sum of two integers a and b.',
    'Authenticate a user given username and password.',
]

results = benchmark.run(strategies, refs)
print('Benchmark results:')
for name, metrics in results.items():
    print(f'  {name}: {metrics}')

In [ ]:
# Bar chart comparison
rows = []
for name, metrics in results.items():
    if isinstance(metrics, dict):
        for metric, val in metrics.items():
            rows.append({'Strategy': name, 'Metric': metric, 'Score': val})

if rows:
    df = pd.DataFrame(rows)
    fig = px.bar(df, x='Metric', y='Score', color='Strategy', barmode='group',
                 title='Benchmark: Strategy Comparison')
    fig.show()
else:
    print('No structured metrics returned — check BenchmarkRunner output format.')